In [1]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from scipy.io import loadmat

## Extract

In [2]:
RAW_DATA_DIR = "..\\data\\01_raw"

In [63]:
# device: (folder_name, visits)
DEVICE_CONFIG = {
    "bia": ("BIA", (1,)),
    "exg": ("EXG", (2, 3)),
    "one_rm": ("1RM", (None,)), # EMG data for normalisation + 1RM table (not needed in analysis)
    "ip": ("IP", (2, 3)),
    "imu": ("IMU", (2, 3)),
    "cosmed": ("COSMED", (2, 3)),
    "lactate": ("Lactate", (2, 3)),
    "markers": ("Markers", (2, 3)),
    "lpt": ("LPT", (1, 2, 3)),
}

extensions = {
        "bia": "csv",
        "exg": "mat",
        "one_rm": "mat",
        "ip": "csv",
        "imu": "csv",
        "cosmed": "xlsx",
        "lactate": "xlsx",
        "markers": "csv",
        "lpt": "csv",
    }

FILE_VARIANTS = {
    "lpt": {
        "raw": r"LPTRaw",
        "processed": r"LPT(?!Raw)",
    }
}

DEFAULT_VARIANT = {
    "lpt": "processed",
}

In [4]:
os.listdir(Path(RAW_DATA_DIR))

['.gitkeep',
 '1RM',
 'BIA',
 'COSMED',
 'EXG',
 'IMU',
 'IP',
 'Lactate',
 'LPT',
 'Markers',
 'participants.db',
 'Participant_Metadata.xlsx']

### Metadata or Static Data

#### Select participant

In [5]:
def load_metadata(num):
    '''
    Input: Text, represents participant number. For volunteers 'V' added at the start
    Returns: Tuple with participant number and the corresponding metadata in the series 
    '''
    metadata = pd.read_excel(Path(RAW_DATA_DIR+"\\Participant_Metadata.xlsx"), engine="openpyxl")
    metadata.set_index("Participant", inplace=True)
    metadata_series = metadata.loc[num]
    metadata_features = metadata_series.index.tolist()
    metadata_array = metadata_series.replace({'M': 0, 'F' : 1, 'Yes' : 1, 'No' : 0}).astype(float).to_numpy().reshape(1, -1)
    return metadata_features, metadata_array

### BIA

In [ ]:
# later use this in another function to load all data together into appropriate dataframe numpy array names
# explore synchronization for all in a separate notebook
# explore each in separate notebooks

In [67]:
def load_participant_data(participant_number, device, visit=None, extension="", variant=None):
    device_key = device.lower()

    if device_key not in DEVICE_CONFIG:
        raise ValueError(
            f"Unknown device {device!r}."
            f"Choose from: {', '.join(DEVICE_CONFIG)}"
        )

    folder_name, expected_visits = DEVICE_CONFIG[device_key]
    data_dir = Path(RAW_DATA_DIR)/folder_name
    
    variant_pattern = FILE_VARIANTS.get(device_key, {}).get(variant, "")
    pattern = re.compile(
        rf"^P{re.escape(participant_number.upper())}"
        rf".*{variant_pattern}.*\.{re.escape(extension)}$",
        re.IGNORECASE
        )
    files = [
        os.path.join(data_dir, filename)
        for filename in os.listdir(data_dir)
        if pattern.match(filename)
        ]

    if not files:
        raise FileNotFoundError(
            f"No {device} file found for participant {participant_number}"
        )

    def file_visit(path):
        match = re.search(r"(?:^|_)V(\d+)(?:_|$)", path, re.IGNORECASE)
        return int(match.group(1)) if match else None

    if visit is None:
        if len(files) != 1:
            available_visits = sorted(
                file_visit(path) for path in files
                if file_visit(path) is not None
            )
            raise ValueError(
                f"{device} has multiple files for {participant_number}. "
                f"Specify visit explicitly. Available visits: {available_visits}"
            )
        selected_file = files[0]
    elif(visit > 3 or visit < 1):
        raise ValueError(
            f"Incorrect visit number specified, visit {visit} "
        )
    else:
        matching_files = [
            path for path in files
            if file_visit(path) == visit
            or (file_visit(path) is None and visit == 1)
        ]

        if len(matching_files) != 1:
            raise FileNotFoundError(
                f"Could not find exactly one {device} file for "
                f"{participant_number}, visit {visit}"
            )

        selected_file = matching_files[0]
    extension = Path(selected_file).suffix.lower()
    if extension == ".csv":
        data = pd.read_csv(selected_file)
    elif extension == ".xlsx":
        data = pd.read_excel(selected_file)
    elif extension == ".mat":
        data = loadmat(selected_file)
    else:
        raise ValueError(f"Unsupported file type: {extension}")
    print("Extension:", extension)
    return data

    

In [82]:
def load_data(participant_number):
    metadata_features, metadata_array = load_metadata(participant_number)

    data = {
        "metadata_features": metadata_features,
        "metadata": metadata_array,
        "devices": {},
    }
    
    for device, (_, visits) in DEVICE_CONFIG.items():
        if device == "lpt":
            for variant in FILE_VARIANTS.get(device, {}).keys():
                data["devices"][device] = {
                    visit: load_participant_data(
                    participant_number,
                    device,
                    visit=visit,
                    extension=extensions[device],
                    variant=variant
                )
                for visit in visits
                }
        else:
            data["devices"][device] = {
                visit: load_participant_data(
                participant_number,
                device,
                visit=visit,
                extension=extensions[device],
            )
            for visit in visits
            }
    return data

In [83]:
participant_data = load_data("V01")

Extension: .csv
Extension: .mat
Extension: .mat
Extension: .mat
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .xlsx
Extension: .xlsx
Extension: .xlsx
Extension: .xlsx
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv
Extension: .csv


In [91]:
participant_data.keys()

dict_keys(['metadata_features', 'metadata', 'devices'])

In [88]:
device = "imu"
visit = 2
participant_data["devices"][device][visit]

,timestamp_us,wall_timestamp,rx_yaw,ry_roll,rz_pitch,ax,ay,az,gx,gy,gz,gravx,gravy,gravz,qw,qx,qy,qz
0,3479207693,2026-09-03T15:32:16.531212,197.5000,15.1875,-79.1875,0.00,0.06,-0.28,1.7500,9.6875,9.2500,2.57,9.29,1.76,-0.1171,-0.2628,-0.5839,-0.7592
1,3479218057,2026-09-03T15:32:16.545447,197.3750,15.2500,-79.2500,-0.03,0.08,-0.21,1.7500,7.3750,8.7500,2.58,9.29,1.76,-0.1153,-0.2629,-0.5840,-0.7593
2,3479228424,2026-09-03T15:32:16.571682,197.2500,15.3125,-79.2500,0.37,-0.15,-0.19,2.5000,4.8750,4.9375,2.60,9.29,1.75,-0.1146,-0.2629,-0.5842,-0.7593
3,3479239433,2026-09-03T15:32:16.571682,197.0625,15.4375,-79.2500,0.40,-0.01,-0.12,3.0000,4.5000,6.1250,2.61,9.28,1.75,-0.1139,-0.2629,-0.5843,-0.7593
4,3479249719,2026-09-03T15:32:16.578292,196.8750,15.5000,-79.2500,0.33,0.11,0.00,3.0625,7.1250,9.3125,2.62,9.28,1.75,-0.1129,-0.2630,-0.5844,-0.7593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369804,2970468902,2026-09-03T16:35:22.821898,236.4375,17.8750,-83.8750,-0.29,0.02,-0.27,-4.0625,-11.1875,-2.8750,3.01,9.28,0.98,-0.3509,-0.4840,-0.4641,-0.6537
369805,2970479026,2026-09-03T16:35:22.821898,236.5625,17.8125,-83.8750,-0.21,-0.14,-0.26,-3.5000,-10.8125,-3.0000,3.00,9.28,0.98,-0.3516,-0.4844,-0.4636,-0.6533
369806,2970489147,2026-09-03T16:35:22.838893,236.6875,17.8125,-83.8750,-0.20,-0.17,-0.26,-2.8125,-10.4375,-2.8750,3.00,9.28,0.98,-0.3523,-0.4847,-0.4633,-0.6529
369807,2970499271,2026-09-03T16:35:22.838893,236.7500,17.8125,-83.8750,-0.17,-0.18,-0.25,-1.8750,-9.5000,-2.8125,3.00,9.28,0.98,-0.3529,-0.4851,-0.4630,-0.6525


In [192]:
load_participant_data("V01", "exg", visit=2, extension="mat")

Extension: .mat


{'__header__': b'MATLAB 5.0 MAT-file, Platform: Win32NT, CREATED on: Wed, 16 Sep 2026 20:29:16 GMT',
 '__version__': '1.0',
 '__globals__': [],
 'SamplingFrequency': array([[2048.]]),
 'Data': array([[array([[ 0.4597982 , -0.03662109],
                [ 0.289917  , -0.02543131],
                [ 0.26194254, -0.02288818],
                ...,
                [ 0.41249594, -0.07273356],
                [ 0.43436685, -0.06561279],
                [ 0.46437582, -0.06917318]], shape=(7815177, 2), dtype=float32)]],
       dtype=object),
 'Time': array([[array([[0.00000000e+00],
                [4.88281250e-04],
                [9.76562500e-04],
                ...,
                [3.81600293e+03],
                [3.81600342e+03],
                [3.81600391e+03]], shape=(7815177, 1))]], dtype=object),
 'Description': array([[array(['Pectoralis Major - IN 1 (Channel 2) - General[mV]'], dtype='<U49')],
        [array(['Latissimus Dorsi - IN 1 (Channel 1) - General[mV]'], dtype='<U49')]],
  

In [173]:
bia.head()

,Value,Unit,"2 Sept 2026, 14:19","2 Sept 2026, 14:16","2 Sept 2026, 14:08"
0,Weight,kg,72.90,72.9,72.9
1,Skeletal Muscle Mass,kg,30.90,NaN,NaN
2,Fat Mass,kg,11.61,NaN,NaN
3,Visceral Adipose Tissue,NaN,NaN,NaN,NaN
4,Segmental Skeletal Muscle Mass,NaN,NaN,NaN,NaN


In [ ]:
device = "IMU"
device_key = device.lower()
folder_name, expected_visits = DEVICE_CONFIG[device_key]
data_dir = Path(RAW_DATA_DIR)/folder_name


In [138]:
filenames = [filename for filename in os.listdir(data_dir)]

In [133]:
pattern

re.compile(r'P^V01.*', re.UNICODE)

In [ ]:
selected_file = 

In [123]:
for path in os.listdir(data_dir):
    print(path)

.gitkeep
P01_20260805_V1_IMU.csv
P01_20260810_V1_T151310_IMU.csv
P01_20260821_V1_T114734_Combined.xlsx
P01_20260821_V1_T114734_IMU.csv
PTest01_20260903_20260903_V1_T135121_IMU.csv
PTest_20260903_VHome_T235815_IMU.csv
PV01_V2_IMU_20260903_T153216.csv
PV01_V3_IMU_20260907_T112437.csv
P_Test_20260831_V_Test_T121413_IMU.csv


In [120]:
re.search(rf"(^|_){re.escape(folder_name)}(_|$)", "data/01_raw/IMU/PV01_V2_IMU_20260903_T153216.csv")

<re.Match object; span=(23, 28), match='_IMU_'>

In [116]:
data_dir

WindowsPath('../data/01_raw/IMU')

In [115]:
files

[WindowsPath('../data/01_raw/IMU/PV01_V2_IMU_20260903_T153216.csv'),
 WindowsPath('../data/01_raw/IMU/PV01_V3_IMU_20260907_T112437.csv')]

In [81]:
list(data_dir.glob(f"P{participant_number.upper()}*.csv"))

[WindowsPath('../data/01_raw/IMU/PV01_V2_IMU_20260903_T153216.csv'),
 WindowsPath('../data/01_raw/IMU/PV01_V3_IMU_20260907_T112437.csv')]

In [109]:
pattern = re.compile(rf"^P{re.escape(participant_number.upper())}(_|$)")

In [110]:
pattern.match("PV01_V2_IMU_20260903_T153216.csv, PV01_V3_IMU_20260907_T112437.csv")

<re.Match object; span=(0, 5), match='PV01_'>

In [112]:
filenames = [
    "PV01_V2_IMU_20260903_T153216.csv",
    "PV01_V3_IMU_20260907_T112437.csv",
]
matches = [pattern.match(filename) for filename in filenames]

In [113]:
matches

[<re.Match object; span=(0, 5), match='PV01_'>,
 <re.Match object; span=(0, 5), match='PV01_'>]

In [ ]:
"PV01_V2_IMU_20260903_T153216.csv"

In [47]:
Path(RAW_DATA_DIR)/folder_name

WindowsPath('../data/01_raw/IMU')

In [61]:
files = [path for path in data_dir.glob(f"P{participant_number.upper()}*.csv")]

In [62]:
files[0].stem

'PV01_V2_IMU_20260903_T153216'

In [50]:
rf"(^|_){re.escape(folder_name)}"

'(^|_)IMU'

In [ ]:
re.search(rf"(^|_){re.escape(folder_name)}", re.IGNO)

TypeError: expected string or bytes-like object, got 'RegexFlag'

In [45]:
participant_number.upper()

'V01'

In [58]:
files

[WindowsPath('../data/01_raw/IMU/PV01_V2_IMU_20260903_T153216.csv'),
 WindowsPath('../data/01_raw/IMU/PV01_V3_IMU_20260907_T112437.csv')]

In [ ]:
bia = pd.read_csv(os.path.relpath(raw + f"\\P{partcipant_id}_{device}